# CHIRPS x OSM

- replacing the placeholder `HIGHWAY_SURFACE_RISK` table used in `graph-representation/03_pulling_real_graph.ipynb`
with a rainfall-aware version.

Recap of the decisions this notebook implements:
- **Accumulation window:** rolling 3-day sum of daily CHIRPS precipitation (mm), per pixel. (To Review - we may need to consider that is accumulation in terms of draining water in different soils)
- **Rain risk thresholds:** `<5mm -> 0.0`, `5-15mm -> 0.3`, `15-30mm -> 0.6`, `>=30mm -> 1.0`.
- **Road rain vulnerability:** how much a given `road_type` is affected by rain
  (paved roads barely, tracks a lot).
- **Final formula:** `surface_risk = min(1.0, base_road_risk + vulnerability[road_type] * rain_risk)`.

In [9]:
%pip install osmnx networkx geopandas shapely folium earthengine-api pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import ee

In [11]:
ee.Authenticate()


Successfully saved authorization token.


In [12]:
ee.Initialize(project='project-0cf410a3-f35f-4911-9c5')

In [13]:
from dataclasses import dataclass, field
from typing import Iterable

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import folium


### Core implementation

**`ParkRoadNetwork`** — pulls the OSM road graph for a park and returns the
vehicle-usable edges with travel times, mirroring the loading/filtering steps from
`graph-representation/03_pulling_real_graph.ipynb`.

**`EdgeRiskEnricher`** —

### Wiring it together

In [21]:
from park_road_network import ParkRoadNetwork

# Same base road-type risk table used in graph-representation/03_pulling_real_graph.ipynb
HIGHWAY_SURFACE_RISK = {
    "motorway": 0.0, "trunk": 0.0, "primary": 0.0,
    "secondary": 0.1,
    "tertiary": 0.2, "residential": 0.2,
    "service": 0.3,
    "unclassified": 0.4,
    "track": 0.8,
}

PLACE_NAME = "Tarangire National Park, Tanzania"
RAIN_WINDOW_START_DATE = "2025-07-01"
RAIN_WINDOW_END_DATE = "2025-07-04"

nodes, edges = ParkRoadNetwork(PLACE_NAME).load()
print(f"Vehicle-usable edges: {len(edges)}")

Vehicle-usable edges: 2207


In [22]:
from chirps_provider import CHIRPSProvider

rain_provider = CHIRPSProvider()
pixel_grid = rain_provider.frame_grid(edges.total_bounds)

rain_pixels = rain_provider.get_accumulated_precipitation(
    pixel_grid,
    start_date=RAIN_WINDOW_END_DATE,
    end_date=RAIN_WINDOW_END_DATE
)

print(f"Pixels: {len(rain_pixels)}")
print(rain_pixels["rain_mm"].describe())

Pixels: 180
count    1.800000e+02
mean     1.572057e-01
std      1.753333e-01
min      4.682976e-09
25%      4.132080e-02
50%      8.025658e-02
75%      2.035463e-01
max      8.576328e-01
Name: rain_mm, dtype: float64


In [23]:
enricher = EdgeRiskEnricher(base_road_risk=HIGHWAY_SURFACE_RISK)
edges_enriched = enricher.enrich(edges, rain_pixels)

print(edges_enriched[[
    "road_type", "travel_time_m", "rain_mm", "rain_risk",
    "base_road_risk", "surface_risk", "cost",
]].head(10))

print("\nsurface_risk distribution:")
print(edges_enriched["surface_risk"].describe())

                             road_type  travel_time_m   rain_mm  rain_risk  \
u         v          key                                                     
460425854 4681959723 0    unclassified       0.022192  0.273888        0.0   
          460425855  0    unclassified       0.036043  0.273888        0.0   
          4681960245 0           track       0.030258  0.273888        0.0   
460425855 460425854  0    unclassified       0.036043  0.273888        0.0   
          4681960245 0           track       0.017009  0.273888        0.0   
          460425857  0    unclassified       0.373986  0.273888        0.0   
460425857 4681960250 0           track       0.347814  0.273888        0.0   
          460425855  0    unclassified       0.373986  0.273888        0.0   
          2196664391 0    unclassified       8.250063  0.292080        0.0   
618532432 4660249470 0           track       7.405159  0.206297        0.0   

                          base_road_risk  surface_risk       co

### Visualizing edge cost

Same cost-bucket coloring as `graph-representation/03_pulling_real_graph.ipynb`,
now driven by rainfall-aware `surface_risk` instead of the fake, road-type-only table.

In [24]:
center_lat, center_lon = nodes["y"].mean(), nodes["x"].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

COST_COLORS = {
    (0, 20): "#2ecc71",
    (20, 40): "#f39c12",
    (40, 80): "#e74c3c",
    (80, 999): "#7b241c",
}

def get_cost_color(cost: float) -> str:
    for (low, high), color in COST_COLORS.items():
        if low <= cost < high:
            return color
    return "#7b241c"

rain_window_days = start

for _, row in edges_enriched.iterrows():
    coords = [(lat, lon) for lon, lat in row["geometry"].coords]
    road_type = row["road_type"]
    rain_mm = row["rain_mm"]
    surface_risk = row["surface_risk"]
    cost = row["cost"]
    folium.PolyLine(
        locations=coords,
        color=get_cost_color(cost),
        weight=3,
        opacity=0.8,
        popup=(
            f"highway: {road_type}<br>"
            f"rain_mm ({RAIN_WINDOW_DAYS}d): {rain_mm:.1f}<br>"
            f"surface_risk: {surface_risk:.2f}<br>"
            f"cost: {round(cost, 1)} min"
        ),
    ).add_to(m)

m


### Next steps

- Validate the rain-risk thresholds and road-vulnerability multipliers against real
  road-condition reports, once available — they're reasoned estimates, not fitted.
- `RAIN_WINDOW_END_DATE` is hardcoded to a date from the eda exploration; wire it to
  "most recent date CHIRPS has published" for anything beyond notebook experimentation.
- Feed `edges_enriched` (with its `cost` column) into a shortest-path routine
  (e.g. `networkx.shortest_path` with `weight="cost"`) to get an actual risk-aware route.